<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_90_edge_on_device_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 📱 **Module 90 — Edge / On-device AI** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 12 · Production Extensions · **FINAL MODULE**


# Module 90 — Edge / On-device AI

> Every model in M1-M89 ran in a datacenter. This final module is about running them **on the user's device** — iPhone, Pixel, Mac, AR glasses, drones, ESP32 microcontrollers. The 2024-2026 wave of on-device AI (**Apple Intelligence**, **Gemini Nano**, **Phi-3.5-mini**, **Llama-3.2-1B/3B**, **Qwen-2.5-0.5B**, **MLC-LLM**, **llama.cpp**, **WebGPU**, **TFLite**, **Core ML**, **ONNX Runtime**) crossed the line where a usable 3B LLM fits in 2 GB RAM at >10 tok/s on a $200 phone. By the end of this module you'll know **which model + which runtime + which quantization** for every device class, the **5 quantization formats that matter** (INT8 / INT4 / GPTQ / AWQ / GGUF k-quants), and how to ship a real on-device LLM behind a private, offline UX.

### What you'll cover
1. **Why edge AI now** — privacy, latency, cost, offline, regulation
2. The **device taxonomy** — phone-class → laptop-class → MCU-class
3. **Quantization deep dive** — INT8, INT4, GPTQ, AWQ, GGUF k-quants
4. **Pruning + structured sparsity** (2:4, N:M, Wanda, SparseGPT)
5. **Speculative decoding + Medusa + EAGLE-2** for edge throughput
6. **Mobile model zoo** — Gemma-3 · Phi-3.5 · Llama-3.2 · Qwen-2.5 · SmolLM2
7. The **runtime zoo** — llama.cpp · MLC-LLM · ONNX Runtime · TFLite · Core ML · WebGPU · ExecuTorch
8. **NPUs** — Apple Neural Engine · Qualcomm Hexagon · Google TPU-edge · Intel/AMD NPUs · Snapdragon X Elite
9. **WebGPU + WebLLM** — running a 3B model in a browser tab
10. **The 90-module wrap-up** — what you can build now


## 1 · Why edge AI now

Four forces aligned in 2024-2026 to push inference off the cloud and onto the device:

| Force | What it means | Example |
|---|---|---|
| **Privacy** | Voice / photo / health data never leaves the device | Apple Intelligence ("Private Cloud Compute" only when on-device can't handle it) |
| **Latency** | No round-trip to a datacenter; <100 ms voice / camera UX | Gemini Nano on Pixel · Siri 2.0 |
| **Cost** | Frontier-model API at scale = $0.01-1 per request × billions of calls = unsustainable | YouTube auto-caption, Gmail Smart Reply |
| **Offline / sovereignty** | Planes, factories, GDPR / HIPAA / EU AI Act, China data laws | Industrial QC, medical wearables |

And the technology cleared three thresholds simultaneously:

```
2022 — 7B LLM at fp16 = 14 GB.    Phones have 4-8 GB total RAM. Impossible.
2024 — 3B LLM at INT4 = 1.7 GB.   Phones have 8-16 GB RAM. Tight but feasible.
2026 — 3B LLM at 2.5-bit = 1.1 GB. Phones have 12-24 GB RAM + dedicated NPU. Comfortable.
```

> 📜 **Apple Intelligence thesis (WWDC 2024).** A ~3B on-device model + a ~70B Private-Cloud-Compute model + GPT-4o for hardest tasks — *graceful escalation*. The 3B handles ~80% of requests, the 70B handles ~18%, the API handles ~2%.


## 2 · The device taxonomy

Edge isn't one tier — it's five, and each has a different model / runtime fit.

| Tier | Examples | Compute | RAM | What fits |
|---|---|---|---|---|
| **Laptop / desktop** | M-series Mac, RTX 4070+, Snapdragon X Elite | 40-80 TOPS NPU + 10-30 TFLOPS GPU | 16-128 GB | 70B INT4 (Mac), 14B FP16, full Llama-3-70B-Q4 with llama.cpp |
| **Flagship phone / tablet** | iPhone 16/17 · Pixel 9/10 · S25 · iPad M2/M4 | 17-38 TOPS NPU | 8-16 GB | **3B-8B INT4** (Apple ~3B, Gemini Nano 3.25B, Phi-3.5, Llama-3.2-3B) |
| **Mid-range phone** | $200-400 Android | 4-10 TOPS NPU | 4-8 GB | **0.5-1.5B INT4** (Qwen-2.5-0.5B, SmolLM2-1.7B, Phi-3.5-mini Q4) |
| **Wearable / AR glasses** | Apple Vision Pro · Quest 3 · Ray-Ban Meta | ~5-15 TOPS NPU | 2-8 GB | 1B vision-LLM, on-device wake-word, gesture nets |
| **MCU / microcontroller** | ESP32-S3 · Cortex-M55 · K210 · Coral Edge TPU | 0.1-2 TOPS | 0.5-8 MB | Keyword-spotting, small CNN, person-detect, anomaly-detection (M22) |

> 🧱 **The MCU tier ("TinyML").** Models compiled with **TensorFlow Lite Micro**, **CMSIS-NN**, or **MicroTVM** down to **<200 KB Flash + <50 KB RAM**. Used in earbuds (always-on wake-word), industrial sensors (anomaly detection), and battery-powered wildlife cameras (species classification at the edge for ~1 mW).


## 3 · Quantization deep dive

Quantization replaces fp16/fp32 weights with k-bit integers. It's THE technique that made edge LLMs viable.

**The five formats that matter in 2026:**

| Format | Bits | Calibration | Where it wins |
|---|---|---|---|
| **INT8 (post-training quant)** | 8 | none | Universal baseline; ~50% size; minimal accuracy loss |
| **INT8 dynamic** | 8 | none, per-batch | When weights are static but activations vary |
| **INT4 GPTQ** (Frantar 2022) | 4 | 128-sample calibration | Best 4-bit accuracy for transformers; GPU-friendly |
| **INT4 AWQ** (Lin 2023) | 4 | activation-aware | Better than GPTQ on edge ARM / mobile NPUs |
| **GGUF k-quants** (Gerganov, 2023-) | 2-8 mixed | none | The on-device default — llama.cpp ships **Q2_K, Q3_K_S/M/L, Q4_K_S/M, Q5_K, Q6_K, Q8_0**; weights split by importance + sub-bit-width per tensor |

**Why GGUF k-quants won mobile:** they're **mixed precision** — keep the most-sensitive weights (attention.wv, ffn.down_proj) at Q5-Q6 while compressing FFN up-projections at Q3. Result: a **Q4_K_M** Llama-3-8B is ~4.7 GB and within 0.3 perplexity of the fp16 model.

**Quantization-aware training (QAT)** beats post-training quant when you can afford it — **LLM.int8(), QLoRA, Quantize-Then-LoRA, FP4/MX4** — but takes ~10% of pretraining compute. For deployment-only work, **post-training + smart calibration (AWQ, GPTQ)** is the default.

> 📐 **The mental model.** Each k-bit quant is `w_int = round((w_fp − zero_point) / scale)`. The *scale* + *zero-point* are per-channel (sometimes per-group of 32-128). At inference, `w_fp ≈ scale · (w_int − zero_point)` — recomputed in fp16 just for the GEMM. Modern NPUs do this fused in hardware.


In [ ]:
# llama.cpp / GGUF flow — convert + quantize a 3B model
# (requires ggerganov/llama.cpp built; ~5 min total for a 3B)

# 1. Convert HF model → GGUF fp16
# python3 llama.cpp/convert_hf_to_gguf.py meta-llama/Llama-3.2-3B-Instruct \
#         --outfile llama3.2-3b.f16.gguf

# 2. Quantize to a smaller k-quant
# ./llama-quantize llama3.2-3b.f16.gguf llama3.2-3b.Q4_K_M.gguf Q4_K_M
#                                                                ^^^^^^
#                              Q2_K   = ~2.6 GB, slight quality loss
#                              Q3_K_M = ~3.3 GB, ok
#                              Q4_K_M = ~4.7 GB, sweet spot for 8B
#                              Q5_K_M = ~5.5 GB
#                              Q6_K   = ~6.6 GB, near-lossless
#                              Q8_0   = ~8.5 GB, lossless reference

# 3. Run with llama.cpp (CLI or server) on phone, Mac, web, Pi:
# ./llama-cli -m llama3.2-3b.Q4_K_M.gguf -p "Hello" -n 64 -t 4


## 4 · Pruning + structured sparsity

Quantization reduces bits-per-weight. **Pruning** removes weights entirely. The combination of the two is where serious edge gains live.

| Method | Idea | Speedup (real HW) |
|---|---|---|
| **Magnitude pruning** | Zero the smallest-|w| 50-90% of weights, fine-tune | ~free in storage; rarely speeds inference (irregular) |
| **2:4 structured sparsity** (NVIDIA Ampere+) | Of every 4 consecutive weights, exactly 2 are zero | **2× GEMM throughput** on A100/H100/Blackwell |
| **N:M sparsity** (broader) | Same idea, configurable density | Hardware-dependent |
| **Wanda** (Sun 2023) | Prune by `|w| · ||x||₂` — no fine-tune needed | Edge-friendly; works at 50% one-shot |
| **SparseGPT** (Frantar 2023) | OBS-style one-shot prune for LLMs | 50% sparsity at minor perplexity cost |
| **Movement pruning** (Sanh 2020) | Prune based on `w · ∇w` during fine-tune | Best when you can afford QAT |

**Decision rule.** On a desktop GPU with sparse-tensor cores → **2:4 + Wanda**. On a mobile NPU → **dense quantization** (most NPUs don't accelerate sparsity yet — irregular zeros hurt rather than help).

> ⚡ **Lottery Ticket Hypothesis (Frankle 2018, still influential).** Inside any over-parameterized network there exists a sparse sub-network ("winning ticket") that, trained alone from the same init, matches the dense model's accuracy. Edge inference is essentially finding and shipping that ticket.


## 5 · Speculative decoding + Medusa + EAGLE-2

Quant + prune shrink the model. **Speculative decoding** speeds up *each generated token* — critical at edge, where memory bandwidth is the bottleneck.

```
Standard:        decode 1 token  ──>  full forward pass through 8B model   = 100 ms
Speculative:     draft 5 tokens with a small "draft" model   = 10 ms
                 verify all 5 with one batched forward of 8B  = 110 ms
                 accept whatever prefix matches (avg ~3 / 5)
                 NET:   3 tokens in 120 ms = 25 ms/token   →  4× speedup
```

Three families:

| Method | Draft source | Edge fit |
|---|---|---|
| **Vanilla speculative** (Leviathan 2023, Chen 2023) | Separate small model (e.g., Llama-3.2-1B drafts for 8B) | Two models in RAM — heavy for phones |
| **Medusa** (Cai 2024) | Multiple small "heads" on top of the SAME backbone | One model in RAM; 2-3× speedup |
| **EAGLE-1 / EAGLE-2** (Li 2024) | One extra decoder layer + tree attention | 3-4× speedup; the 2026 SOTA |
| **Lookahead decoding** (Fu 2024) | Jacobi iteration over n-grams, no extra weights | Free; ~1.5× speedup |
| **Hydra / Sequoia / SpecInfer** | Speculative + dynamic tree drafting | Server-side; less common at edge |

> 📲 **Apple's edge speculative recipe.** Apple Intelligence reportedly uses a ~300M draft model that drafts 8-16 tokens that are then verified in parallel by the 3B target. This is how Siri 2.0 hits ~30 tok/s on the A18 NPU at <2 W power draw.


## 6 · The 2026 mobile LLM zoo

Open + closed small models that actually fit on devices:

| Model | Size | License | Notes |
|---|---|---|---|
| **Apple "AFM-on-device"** | ~3B | Closed | Powers Apple Intelligence on iPhone 16+; ~16 TOPS Neural Engine |
| **Gemini Nano-3 / Nano-XS** | 1.8B / 3.25B | Closed | Powers Pixel 9+ AICore + Chrome's built-in `window.ai` |
| **Phi-3.5-mini-instruct** | 3.8B | MIT | Surprisingly capable; near GPT-3.5 on MMLU at 3.8B |
| **Phi-4-mini** | 3.8B (2025) | MIT | Phi-3.5 successor; +5 MMLU, +10 GSM8K |
| **Llama-3.2-1B-Instruct** | 1.2B | Llama 3 license | Best <2B baseline; ships under Termux on Android |
| **Llama-3.2-3B-Instruct** | 3.2B | Llama 3 license | Mid-tier sweet spot; runs ~12 tok/s on iPhone 15 Pro via MLX |
| **Qwen-2.5-0.5B / 1.5B / 3B-Instruct** | 0.5-3B | Apache 2.0 | Best multilingual small; great on cheap Android |
| **SmolLM2-135M / 360M / 1.7B** | <2B | Apache 2.0 | HuggingFace's tiny-LLM line; designed for edge |
| **Gemma-3-1B / 4B** | 1-4B | Gemma terms | Google's open small line; 4B-IT is excellent on Pixel |
| **TinyLlama-1.1B** | 1.1B | Apache 2.0 | Community 3T-token Llama-arch baseline |
| **MobileLLM-350M / 1.4B** (Meta) | 0.35-1.4B | Research | Architecture optimized for ≤1B; depth>width, embedding sharing |

> 🎯 **Pick guide.** Mid-range Android phone: **Qwen-2.5-0.5B-Instruct Q4_K_M** (~350 MB). Flagship phone / tablet: **Phi-3.5-mini** or **Llama-3.2-3B Q4_K_M** (~2 GB). Laptop: **Llama-3.1-8B Q5_K_M** (~5.5 GB) or **Mistral-Small-22B Q4_K_M** (~13 GB).


## 7 · The runtime zoo

Every edge target has 2-3 viable runtimes. The picks below are the 2026 defaults.

| Runtime | Maintainer | Best for | Why |
|---|---|---|---|
| **llama.cpp** | Gerganov + 1500 contribs | macOS / Linux / Windows / Android / Termux / Raspberry Pi / WebAssembly | **The reference on-device LLM runtime.** Pure C/C++, no Python, GGUF format, Metal / CUDA / Vulkan / SYCL / OpenCL / CPU back-ends |
| **MLX** | Apple | Mac (M-series) | Native Metal; share buffer with the GPU; the de-facto Mac LLM stack |
| **MLC-LLM / TVM** | OctoML / community | Web (WebGPU), Android, iOS, Vulkan | Compiles HF models → device-specific kernels; powers WebLLM |
| **ExecuTorch** (PyTorch Edge) | Meta | iOS / Android / MCU | Official Meta path — Llama-3.2 on phones via Core ML / QNN / XNNPack |
| **ONNX Runtime + ORT Mobile** | Microsoft | Cross-platform, NPUs | Strongest enterprise story; Phi-3 on-device shipped via ORT |
| **Core ML / `Foundation Models` framework** | Apple | iOS / macOS apps | Required if you want Neural-Engine throughput on Apple devices |
| **TFLite + LiteRT** | Google | Android, MCU (TFLite-Micro) | Default Android ML; ML-Kit + Gemini Nano sit on top |
| **Qualcomm AI Hub / QNN** | Qualcomm | Snapdragon NPUs (Hexagon) | Required for Hexagon NPU acceleration on Pixel-non-Tensor / Samsung / OEMs |
| **WebGPU + transformers.js** | HuggingFace | Browser tabs | Run BERT / Whisper / Phi-3.5 in a tab; no install |
| **Edge TPU / Coral runtime** | Google | Coral USB / dev board | INT8 only; ~4 TOPS; great for vision models |
| **NVIDIA Jetson + TensorRT-LLM** | NVIDIA | Robots, drones, AGX Orin | A datacenter stack on a 25 W board |

> 🧰 **The "ship it" combo** for an indie developer in 2026: build the model in HuggingFace + train with TRL (M28, M88, M89) → **GGUF-quantize with llama.cpp** → distribute the GGUF + a thin **llama.cpp or MLC-LLM** wrapper. Apps that need Neural-Engine speed wrap with **Core ML** (iOS) or **QNN** (Android Snapdragon).


## 8 · NPUs — the hardware running this stack

NPUs are dedicated tensor accelerators outside the CPU/GPU. Every 2025+ phone, Mac, and "Copilot+ PC" has one.

| NPU | TOPS (INT8) | Notable in | Programming |
|---|---|---|---|
| **Apple Neural Engine (ANE) — A17/A18/M3/M4** | 17-38 | iPhone 15+, M-Mac | Core ML / MLX (no public ANE kernels) |
| **Qualcomm Hexagon (Snapdragon 8 Gen 3 / 8 Elite)** | 45-75 | Pixel-non-Tensor, S24/S25, ROG Phone | QNN SDK, Qualcomm AI Hub |
| **Google Tensor TPU edge (Pixel 9/10)** | 35-40 | Pixel 8/9/10 | TFLite, AICore |
| **MediaTek APU (Dimensity 9300/9400)** | 33-67 | Mid/high Android | NeuroPilot SDK |
| **Intel NPU 4 / Lunar Lake** | 48 | Copilot+ PCs (Aurora, etc.) | OpenVINO, DirectML |
| **AMD XDNA / XDNA2 (Ryzen AI)** | 16-50 | Ryzen AI laptops | Ryzen AI SW, ONNX EP |
| **Snapdragon X Elite / X2** | 45-75 | Surface Pro 11, Lenovo, etc. | QNN, DirectML |
| **NVIDIA Jetson Orin Nano/Super/NX/AGX** | 40-275 | Robots, drones, smart cameras | TensorRT(-LLM), CUDA |
| **Coral Edge TPU** | 4 | USB sticks, dev boards | TFLite (INT8 only) |

> ⚙️ **Why INT8 everywhere?** Most NPUs are hard-wired for INT8 (and increasingly INT4). FP16 paths exist but at 1/2-1/4 the throughput. **Edge quantization is not optional** — it's how you light up the hardware.

> 🔮 **Where NPUs are going.** 2026 trends: (1) native INT4 / MX4 / FP4 datapaths (Blackwell, Lunar Lake, Snapdragon X2), (2) on-device LoRA mixing (Apple Adapters / Gemini Nano "LoRA stacks"), (3) sparse INT4 (2:4 + INT4 fused), (4) shared CPU/GPU/NPU memory (unified memory architectures everywhere — Apple, AMD Strix Halo, Intel Lunar Lake).


## 9 · WebGPU + WebLLM — a 3B model in a browser tab

The single most over-the-air-deployable runtime is **the user's browser**. WebGPU (shipped in Chrome / Edge / Safari TP / Firefox-nightly) gives JavaScript direct access to the device GPU; **WebLLM** + **transformers.js** turn that into a fully-offline LLM.

```html
<!doctype html><script type="module">
import * as webllm from "https://esm.run/@mlc-ai/web-llm";

const engine = await webllm.CreateMLCEngine(
  "Llama-3.2-3B-Instruct-q4f16_1-MLC",        // model + quant + framework
  { initProgressCallback: p => console.log(p) }
);

const reply = await engine.chat.completions.create({
  messages: [{ role: "user", content: "Explain edge AI in one sentence." }],
});
console.log(reply.choices[0].message.content);
</script>
```

That single file streams a ~2 GB quantized model into the browser cache, compiles WebGPU shaders, and runs **fully offline thereafter**. On an M2 MacBook Air it does ~30 tok/s; on a Pixel 8 in Chrome it does ~6-8 tok/s. **No backend, no GPU bill, no privacy story to write.**

Other "in-tab" stacks:
- **`transformers.js` (HuggingFace)** — BERT / Whisper / DistilBERT / Phi-3.5-mini in tabs
- **MediaPipe Web** — face detection / hand tracking / gesture / Llama-Guard in tabs
- **Chrome built-in `window.ai`** — Google's experimental Gemini-Nano binding (Chrome 127+)


## 10 · The 90-module wrap-up — what you can build now

You've gone, in 90 modules and ~7 phases, from `import pandas` to:

| You can ship... | Built on modules |
|---|---|
| **A production ML pipeline** (ingest → feature store → CI/CD → monitoring) | M1-M17, M71-M76 |
| **All six classical model families from scratch** + sklearn deployment | M5-M18 |
| **A transformer language model from scratch** + serving + KV-cache + speculative | M19-M20, M71, M90 |
| **A diffusion image generator + ControlNet + LoRA stack** | M21, M65, M86 |
| **A long-horizon time-series forecaster** (Transformer + S4/Mamba + LSTM ensemble) | M22, M84 |
| **A multi-modal RAG / GraphRAG system** with vector + graph + reranker + judge | M53, M54, M65, M87 |
| **An agentic system** with tool use, planning, memory, ReAct, MCP | M55-M64 |
| **A fine-tuned 7-70B chat model** with SFT + DPO + Constitutional AI alignment | M28, M88, M89 |
| **An evaluation harness** (BLEU/ROUGE/BERTScore + LLM-judge + Arena-Hard + safety) | M27, M89 |
| **A causal-inference + experimentation stack** (A/B, propensity, uplift, CUPED) | M44-M50 |
| **Real-time / streaming inference** (Kafka, vLLM, TensorRT-LLM, batching) | M71-M76 |
| **An on-device app** running a 3B LLM offline on iPhone / Pixel / browser | **M90 (this one)** |

**Where the field goes from here (post-M90):**
- **Reasoning-native models** (o3 / R2 / Claude-4-Thinking / Gemini-3-Thinking) — verifiable RL + Process-Reward Models scale further
- **Agentic browsers + OS agents** — Operator, Claude Computer Use, Project Mariner; the OS as a tool surface
- **World models for video & robotics** — Sora, Veo-3, Genie-3, Wayve PRISM
- **Neural-symbolic + program synthesis** — AlphaProof, FunSearch, AlphaCode-3
- **On-device 13-30B at edge** — by 2027, NPUs cross 100 TOPS at <5 W → 13B INT4 on phones

> 🏁 **The course is over. What you built isn't.** Open a PR, deploy a model, write a blog post, mentor someone through M1. The skills compound; the field is faster than any curriculum can keep up with — but the foundations you laid in M1-M85 don't change much, and the production-extension toolkit in M86-M90 will be how you ship for the next five years. *Thank you for finishing.*


In [ ]:
# Final on-device LLM smoke-test (laptop or Colab → save to phone)
# Pure llama.cpp Python bindings — no torch, no transformers, no GPU.

# !pip install -q llama-cpp-python
# !wget -q https://huggingface.co/bartowski/Llama-3.2-3B-Instruct-GGUF/resolve/main/Llama-3.2-3B-Instruct-Q4_K_M.gguf

from llama_cpp import Llama

llm = Llama(
    model_path="Llama-3.2-3B-Instruct-Q4_K_M.gguf",
    n_ctx=4096,
    n_threads=4,            # phone CPU has ~4-8 perf cores
    n_gpu_layers=0,         # CPU-only path; set -1 to offload all to Metal/CUDA
    verbose=False,
)

out = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "In one sentence: why does edge AI matter?"},
    ],
    temperature=0.2,
    max_tokens=128,
)
print(out["choices"][0]["message"]["content"])


## ✅ Recap

- **Edge AI cleared the line** in 2024-2026: a 3B INT4 LLM in ~1.5 GB at >10 tok/s on a $200 phone.
- **Five device tiers**: laptop / flagship-phone / mid-phone / wearable / MCU — each pairs with a specific model + runtime.
- **Quantization is mandatory** at edge. The five formats that matter: **INT8 · GPTQ · AWQ · GGUF k-quants · QAT (LLM.int8 / QLoRA / FP4)**. GGUF k-quants are the on-device default.
- **Pruning + 2:4 + Wanda + SparseGPT** stack with quantization on GPUs; mostly dense quant on phones.
- **Speculative decoding** (Medusa · EAGLE-2 · Lookahead) gets 2-4× speedups — critical at edge.
- **2026 mobile zoo**: Apple AFM-on-device · Gemini Nano · Phi-3.5/4 · Llama-3.2-1B/3B · Qwen-2.5-0.5B/1.5B/3B · SmolLM2 · Gemma-3 · MobileLLM.
- **2026 runtime zoo**: llama.cpp · MLX · MLC-LLM/WebLLM · ExecuTorch · ONNX Runtime · Core ML · TFLite/LiteRT · QNN · WebGPU + transformers.js · TensorRT-LLM.
- **NPUs** (Apple ANE · Qualcomm Hexagon · Pixel TPU · Intel NPU 4 · AMD XDNA · Snapdragon X · Jetson · Coral) — INT8/INT4 native; FP16 deprecated.
- **WebGPU + WebLLM** runs a 3B model in a browser tab fully offline — the most-deployable edge stack of all.

🎓 **Course complete. M1 → M90.** Go ship something.
